# B13. ML Tool `predict_risk(customer_id)`

Công cụ để Agent gọi khi câu hỏi cần điểm rủi ro của một hồ sơ. Hàm nằm ở `agent/src/ml_tool.py` (Agent import được; notebook không import được), tên yếu tố tiếng Việt và cách hiển thị giá trị ở `ml/data/ten_dac_trung.csv`.
Notebook này chỉ **kiểm tra** công cụ: gọi hàm trên toàn bộ 61.503 hồ sơ kiểm tra và đối chiếu với kết quả B11, B12; không học hay chọn gì thêm.

## 0. Nguồn tra cứu và cái áp dụng

| Nguồn | Áp dụng | Không áp dụng |
|---|---|---|
| QĐ-26 AT Credit (Điều 2, 3, 5, 7, 8) | Điểm = xác suất đã hiệu chỉnh; dải xếp theo điểm làm tròn 1 chữ số; tối đa 5 yếu tố tăng, 5 yếu tố giảm **kèm giá trị trong hồ sơ**; ghi phiên bản mô hình; trợ lý chỉ nhận mã hồ sơ, không cung cấp kết quả trả nợ | – |
| CFPB, Circular 2022-03 (2022): mô hình phức tạp không miễn nghĩa vụ nêu lý do cụ thể, chính xác | Yếu tố phải là lý do thật của mô hình cho đúng hồ sơ đó (SHAP từng hồ sơ), có tên dễ hiểu; không thay bằng câu chung chung "không đạt điểm" | Quy định Mỹ (ECOA, Reg B) không áp dụng cho AT Credit; chỉ dùng làm thực hành chuẩn của ngành |
| Upstart, *Adverse Action Notices* (tài liệu cho bên cho vay): dùng SHAP từng hồ sơ để chọn lý do, kiểm tra chiều ảnh hưởng | Chọn yếu tố theo SHAP của chính hồ sơ, tách hai chiều tăng / giảm | Gộp các đặc trưng tương quan thành nhóm lý do: QĐ-26 Điều 5 quy định yếu tố là từng dữ liệu kèm giá trị; B6 đã bỏ cặp Spearman ≥ 0,9 |
| Google, *Function calling with the Gemini API*, mục Best practices | Tên hàm rõ, mô tả hàm và tham số cụ thể; tham số kiểu chặt (`customer_id: int`); kiểm tra đầu vào trước khi chạy; xử lý lỗi bằng kết quả có thông báo | – |
| LangGraph, `ToolNode` (tài liệu tham khảo): lỗi của tool trả về mô hình dưới dạng `ToolMessage` | Mã không hợp lệ trả `{"error": ...}` thay vì ném ngoại lệ, để Agent đọc được và trả lời người dùng | – |
| Lundberg, Lee (2017); LightGBM `predict(pred_contrib=True)` | SHAP tính trực tiếp bằng `pred_contrib` của LightGBM; B12 đã chứng minh trùng tuyệt đối TreeSHAP | Không cần thư viện `shap` khi chạy công cụ |

## 1. Luật của công cụ (đặt trước khi chạy kiểm tra)

- **Đầu vào:** một mã hồ sơ (`SK_ID_CURR`). Chỉ nhận hồ sơ thuộc tập kiểm tra (mô hình chưa học); mã khác, mã không phải số, trả `{"error": ...}`. Không nhận dữ liệu nào khác (khoản 4 Điều 8 QĐ-26).
- **Tính trực tiếp từ mô hình** mỗi lần gọi: dòng đặc trưng của hồ sơ (`features.parquet`, cột `TARGET` bị bỏ ngay khi đọc) → pipeline `LGBM.joblib` → xác suất và SHAP (`pred_contrib`, log-odds). Không tra bảng tính sẵn.
- **Điểm:** xác suất × 100, làm tròn 1 chữ số; LightGBM không hiệu chỉnh (B9), công cụ kiểm tra lại điều này khi nạp. **Dải** xếp theo điểm đã làm tròn: dưới 5 thấp, dưới 15 trung bình, còn lại cao.
- **Yếu tố:** tối đa 5 đặc trưng SHAP > 0 lớn nhất (làm tăng) và 5 đặc trưng SHAP < 0 nhỏ nhất (làm giảm), mỗi yếu tố có tên tiếng Việt, giá trị trong hồ sơ (tiền đổi sang đồng, 1 đơn vị = 24 đồng; tỷ lệ ghi %; ô trống ghi "không có dữ liệu"; giá trị chữ giữ nguyên mã gốc của dữ liệu) và mức ảnh hưởng (log-odds, 3 chữ số).
- **Phiên bản mô hình:** `LGBM-` + 12 ký tự đầu sha256 của file mô hình (khoản 5 Điều 7 QĐ-26): file mô hình đổi thì phiên bản đổi.
- **Không lộ nhãn:** đầu ra chỉ có 7 khóa cố định, không khóa nào chứa kết quả trả nợ (khoản 3 Điều 8 QĐ-26).

Tiêu chí đạt: trên mọi hồ sơ kiểm tra, điểm trùng xác suất B11 làm tròn, dải trùng, yếu tố và mức ảnh hưởng trùng SHAP tính hàng loạt; mọi mã ngoài tập kiểm tra bị từ chối.

In [1]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
import sys
sys.path.insert(0, str(ROOT))
from agent.src.ml_tool import _gia_tri, _load, dai_rui_ro, predict_risk

DATA = ROOT / "ml" / "data"
f = pd.read_parquet(DATA / "features.parquet").merge(pd.read_csv(DATA / "split.csv"), on="SK_ID_CURR")
te, tr = f[f.tap == "test"], f[f.tap == "train"]
X_te = te.drop(columns=["SK_ID_CURR", "TARGET", "tap"]).set_index(te.SK_ID_CURR)
du_doan = pd.read_parquet(DATA / "du_doan_test.parquet").set_index("SK_ID_CURR")
ten = pd.read_csv(DATA / "ten_dac_trung.csv", index_col="cot", keep_default_na=False)

X_tool, _, pipe, phien_ban = _load()
print(f"Công cụ nạp {len(X_tool):,} hồ sơ, {X_tool.shape[1]} đặc trưng; có cột TARGET: {'TARGET' in X_tool.columns}; phiên bản {phien_ban}")
assert X_tool.index.equals(X_te.index) and X_tool.columns.equals(X_te.columns)

Công cụ nạp 61,503 hồ sơ, 124 đặc trưng; có cột TARGET: False; phiên bản LGBM-816cbfce6a50


## 2. Bảng tên yếu tố

In [2]:
print(f"{len(ten)} dòng; trùng tên cột với mô hình: {set(ten.index) == set(X_te.columns)}; tên yếu tố trùng nhau: {ten.ten.duplicated().sum()}")
print(ten.dang.value_counts().to_string())
# một ví dụ mỗi cách hiển thị, lấy từ hồ sơ kiểm tra đầu tiên có giá trị
vd = []
for dang, g in ten.groupby("dang"):
    c = g.index[0]
    v = X_te[c].dropna().iloc[0]
    vd.append({"cách": dang, "cột": c, "tên": g.ten.iloc[0], "giá trị gốc": v, "hiển thị": _gia_tri(v, dang, g.don_vi.iloc[0])})
vd.append({"cách": "ô trống", "cột": "EXT_SOURCE_1", "tên": ten.at["EXT_SOURCE_1", "ten"], "giá trị gốc": np.nan, "hiển thị": _gia_tri(np.nan, "diem", "")})
pd.DataFrame(vd).set_index("cách")

124 dòng; trùng tên cột với mô hình: True; tên yếu tố trùng nhau: 0
dang
so      48
co      24
diem    16
chu     13
tien     9
pct      9
ngay     3
yn       2


,cột,tên,giá trị gốc,hiển thị
cách,,,,
chu,NAME_CONTRACT_TYPE,Loại khoản vay đề nghị,Cash loans,Cash loans
co,FLAG_EMP_PHONE,Có khai điện thoại của người sử dụng lao động,1,có
diem,REGION_POPULATION_RELATIVE,Mật độ dân số tương đối của khu vực nơi ở (chu...,0.035792,"0,036"
ngay,DAYS_REGISTRATION,Lần thay đổi đăng ký cư trú gần nhất,-1213.0,1.213 ngày trước ngày nộp hồ sơ
pct,PTI,PTI (tiền trả hằng tháng của khoản đề nghị / t...,0.241526,"24,2%"
so,CNT_CHILDREN,Số con,1,1 người
tien,AMT_INCOME_TOTAL,Thu nhập tháng khách hàng tự khai,171000.0,"4,1 triệu đồng"
yn,FLAG_OWN_CAR,Có ô tô,Y,có
ô trống,EXT_SOURCE_1,Điểm tham khảo bên ngoài số 1 (0 – 1),NaN,không có dữ liệu


## 3. Kết quả tham chiếu tính hàng loạt

Xác suất từ B11 (`du_doan_test.parquet`), SHAP tính một lần cho cả tập bằng `pred_contrib`; công cụ tính từng hồ sơ một, độc lập với phần này.

In [3]:
Xp = pipe[0].transform(X_te)
S = pd.DataFrame(pipe[-1].booster_.predict(Xp, pred_contrib=True)[:, :-1], index=X_te.index, columns=Xp.columns)
diem_ref = np.round(du_doan.loc[X_te.index, "LGBM_hc"].to_numpy() * 100, 1)
print("Xác suất pipeline nạp lại so với B11, lệch lớn nhất:", np.abs(pipe[-1].predict_proba(Xp)[:, 1] - du_doan.loc[X_te.index, "LGBM_hc"]).max())

Xác suất pipeline nạp lại so với B11, lệch lớn nhất: 0.0


## 4. Gọi công cụ trên toàn bộ tập kiểm tra

In [4]:
t = time.time()
ket_qua = {ma: predict_risk(int(ma)) for ma in X_te.index}
giay = time.time() - t
print(f"{len(ket_qua):,} lần gọi, {giay / 60:.1f} phút, trung bình {giay / len(ket_qua) * 1000:.1f} ms/lần")

61,503 lần gọi, 27.1 phút, trung bình 26.4 ms/lần


In [5]:
KHOA = {"customer_id", "risk_score_pct", "risk_band", "factors_increasing", "factors_decreasing", "model_version", "note"}
loi = {"khóa khác mẫu": 0, "điểm lệch B11": 0, "dải lệch": 0, "yếu tố tăng lệch": 0, "yếu tố giảm lệch": 0, "thiếu yếu tố (< 5)": 0}
for i, (ma, r) in enumerate(ket_qua.items()):
    s = S.loc[ma]
    loi["khóa khác mẫu"] += set(r) != KHOA
    loi["điểm lệch B11"] += r["risk_score_pct"] != diem_ref[i]
    loi["dải lệch"] += r["risk_band"] != dai_rui_ro(diem_ref[i])
    for khoa, chon in [("tăng", s[s > 0].nlargest(5)), ("giảm", s[s < 0].nsmallest(5))]:
        ky_vong = [(ten.at[c, "ten"], round(float(v), 3)) for c, v in chon.items()]
        thuc = [(y["factor"], y["impact"]) for y in r[f"factors_{'increasing' if khoa == 'tăng' else 'decreasing'}"]]
        loi[f"yếu tố {khoa} lệch"] += thuc != ky_vong
        loi["thiếu yếu tố (< 5)"] += len(thuc) < 5
pd.Series(loi, name="số hồ sơ").to_frame()

,số hồ sơ
khóa khác mẫu,0
điểm lệch B11,0
dải lệch,0
yếu tố tăng lệch,0
yếu tố giảm lệch,0
thiếu yếu tố (< 5),0


In [6]:
diem = np.array([r["risk_score_pct"] for r in ket_qua.values()])
q = du_doan.loc[X_te.index, "LGBM_hc"].to_numpy() * 100
dai = pd.Series([r["risk_band"] for r in ket_qua.values()])
print(dai.value_counts().reindex([dai_rui_ro(1), dai_rui_ro(10), dai_rui_ro(20)]).to_frame("số hồ sơ").assign(tỷ_lệ=lambda d: d["số hồ sơ"] / len(dai)).to_string())
print(f"\nHồ sơ có điểm chưa làm tròn trong [4,95; 5) hoặc [14,95; 15), được đẩy lên dải trên nhờ làm tròn: "
      f"{((q >= 4.95) & (q < 5)).sum()} và {((q >= 14.95) & (q < 15)).sum()}")
print("Phiên bản trên mọi kết quả:", {r["model_version"] for r in ket_qua.values()})
print("Mọi kết quả chuyển được sang JSON:", all(json.dumps(r, ensure_ascii=False) for r in ket_qua.values()))

                              số hồ sơ     tỷ_lệ
thấp (dưới 5%)                   31967  0.519763
trung bình (5% đến dưới 15%)     20703  0.336618
cao (từ 15%)                      8833  0.143619

Hồ sơ có điểm chưa làm tròn trong [4,95; 5) hoặc [14,95; 15), được đẩy lên dải trên nhờ làm tròn: 246 và 44
Phiên bản trên mọi kết quả: {'LGBM-816cbfce6a50'}


Mọi kết quả chuyển được sang JSON: True


## 5. Mã ngoài tập kiểm tra và đầu vào sai kiểu

In [7]:
ma_train = tr.SK_ID_CURR.to_numpy()
tu_choi_train = sum("error" in predict_risk(int(m)) for m in ma_train)
print(f"Mã thuộc tập huấn luyện bị từ chối: {tu_choi_train:,} / {len(ma_train):,}")
ap_test = te.SK_ID_CURR.iloc[0]
for vao in [0, -1, 999999, "abc", None, 1.5, str(ap_test), f" {ap_test} "]:
    r = predict_risk(vao)
    print(f"{vao!r:>10} →", r.get("error") or f"điểm {r['risk_score_pct']} (mã {r['customer_id']})")

Mã thuộc tập huấn luyện bị từ chối: 246,008 / 246,008
         0 → Không tra cứu được mã hồ sơ 0: mã không có trong danh sách hồ sơ được tra cứu.
        -1 → Không tra cứu được mã hồ sơ -1: mã không có trong danh sách hồ sơ được tra cứu.
    999999 → Không tra cứu được mã hồ sơ 999999: mã không có trong danh sách hồ sơ được tra cứu.
     'abc' → Không tra cứu được mã hồ sơ abc: mã không có trong danh sách hồ sơ được tra cứu.
      None → Không tra cứu được mã hồ sơ None: mã không có trong danh sách hồ sơ được tra cứu.
       1.5 → Không tra cứu được mã hồ sơ 1.5: mã không có trong danh sách hồ sơ được tra cứu.
  '100009' → điểm 0.7 (mã 100009)
' 100009 ' → điểm 0.7 (mã 100009)


## 6. Ba hồ sơ ví dụ (cùng ba hồ sơ của B12, mỗi dải một hồ sơ)

Đây là đúng nội dung Agent nhận được.

In [8]:
for ma in [172588, 292129, 292498]:
    print(json.dumps(predict_risk(ma), ensure_ascii=False, indent=1), "\n")

{
 "customer_id": 172588,
 "risk_score_pct": 2.5,
 "risk_band": "thấp (dưới 5%)",
 "factors_increasing": [
  {
   "factor": "Số ngày trả đủ trước hạn trung bình mỗi kỳ tại AT Credit",
   "value": "1,3 ngày",
   "impact": 0.098
  },
  {
   "factor": "Điểm tham khảo bên ngoài số 2 (0 – 1)",
   "value": "0,485",
   "impact": 0.093
  },
  {
   "factor": "Địa chỉ thường trú khác thành phố với địa chỉ liên hệ",
   "value": "có",
   "impact": 0.092
  },
  {
   "factor": "Số ngày quá hạn lớn nhất của kỳ đến hạn trong 12 tháng gần nhất tại AT Credit",
   "value": "không có dữ liệu",
   "impact": 0.076
  },
  {
   "factor": "Số tiền quá hạn lớn nhất từng có trên CIC",
   "value": "381 nghìn đồng",
   "impact": 0.076
  }
 ],
 "factors_decreasing": [
  {
   "factor": "Điểm tham khảo bên ngoài số 1 (0 – 1)",
   "value": "0,742",
   "impact": -0.517
  },
  {
   "factor": "Loại tổ chức nơi làm việc",
   "value": "Security Ministries",
   "impact": -0.262
  },
  {
   "factor": "Dư nợ / số tiền vay ban

## 7. Nhận xét (sau thực nghiệm)

- **Đạt mọi tiêu chí mục 1.** Trên 61.503 hồ sơ kiểm tra, công cụ gọi từng hồ sơ cho điểm, dải, tên và mức ảnh hưởng của 5 + 5 yếu tố trùng hoàn toàn với kết quả tính hàng loạt (0 hồ sơ lệch); hồ sơ nào cũng đủ 5 yếu tố mỗi chiều. Cả 246.008 mã của tập huấn luyện và mọi đầu vào sai kiểu đều bị từ chối bằng cùng một thông báo; mã dạng chữ số ("100009", có khoảng trắng) được chấp nhận vì mô hình ngôn ngữ có thể truyền mã dưới dạng chuỗi.
- **Không lộ nhãn:** dữ liệu công cụ nạp không có cột `TARGET`; đầu ra chỉ có 7 khóa cố định.
- **Dải:** thấp 52,0%, trung bình 33,7%, cao 14,4% hồ sơ kiểm tra. Quy tắc làm tròn trước khi xếp dải (khoản 2 Điều 3 QĐ-26) đẩy 246 hồ sơ (điểm chưa làm tròn trong [4,95%; 5%)) lên dải trung bình và 44 hồ sơ ([14,95%; 15%)) lên dải cao.
- **Thời gian:** trung bình 26,4 ms mỗi lần gọi sau lần nạp đầu (khoảng 1 giây), không đáng kể so với một lần gọi mô hình ngôn ngữ.
- **Hạn chế của phần hiển thị:** giá trị của cột chữ giữ nguyên mã tiếng Anh của dữ liệu gốc (ví dụ "Drivers", "Business Entity Type 3"); 11 cờ giấy tờ chỉ ghi được "giấy tờ loại k" và ba điểm tham khảo bên ngoài chỉ ghi được số thứ tự, vì Home Credit không công bố giấy tờ và nguồn điểm cụ thể; tỷ lệ làm tròn 1 chữ số (vay / giá hàng 1,000 hiển thị "1 lần").
- Công cụ chỉ giải thích điểm của mô hình, không kiểm tra các ngưỡng chính sách (PTI, DTI, nhóm nợ, thâm niên); các ngưỡng này nằm trong văn bản, Agent tra bằng `search_policy` (notebook 12).